In [9]:
import pathlib as Path
import re
import pandas as pd
from difflib import SequenceMatcher


In [22]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent.parent

DATA_PROCESSED = REPO_ROOT / "data" / "processed"
PERSONAL_DIR = DATA_PROCESSED / "personal_atlas"

league_path = PERSONAL_DIR / "bts_song_league.parquet"
atlas_path = DATA_PROCESSED / "song_embeddings.parquet"

bts_song_league = pd.read_parquet(league_path)
atlas_df = pd.read_parquet(atlas_path)

bts_song_league.shape, atlas_df.shape


((344, 8), (381, 388))

In [14]:
def normalize_title(title: str) -> str:
    if pd.isna(title):
        return ""

    title = str(title).lower()

    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"\[.*?\]", "", title)

    remove_terms = [
        "remastered",
        "remaster",
        "explicit",
        "clean version",
        "instrumental",
        "karaoke",
    ]

    for term in remove_terms:
        title = title.replace(term, "")

    title = title.replace("&", "and")
    title = re.sub(r"[^a-z0-9가-힣ぁ-んァ-ン一-龯\s]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title


bts_song_league["normalized_history_title"] = bts_song_league[
    "master_metadata_track_name"
].apply(normalize_title)

atlas_df["normalized_atlas_title"] = atlas_df["track_name"].apply(normalize_title)

In [15]:
exact_matches = bts_song_league.merge(
    atlas_df[["track_id", "track_name", "album_name", "release_year", "normalized_atlas_title"]],
    left_on="normalized_history_title",
    right_on="normalized_atlas_title",
    how="left",
)

exact_matches["match_type"] = exact_matches["track_id"].notna().map(
    {True: "exact", False: None}
)

exact_matches["track_id"].notna().mean()

np.float64(0.837248322147651)

In [16]:
atlas_titles = atlas_df[["track_id", "track_name", "album_name", "release_year", "normalized_atlas_title"]].drop_duplicates()

def best_fuzzy_match(query: str, choices: pd.DataFrame) -> dict:
    best = None
    best_score = 0

    for _, row in choices.iterrows():
        candidate = row["normalized_atlas_title"]
        score = SequenceMatcher(None, query, candidate).ratio()

        if score > best_score:
            best_score = score
            best = row

    if best is None:
        return {
            "matched_track_id": None,
            "matched_track_name": None,
            "matched_album_name": None,
            "matched_release_year": None,
            "fuzzy_score": 0,
        }

    return {
        "matched_track_id": best["track_id"],
        "matched_track_name": best["track_name"],
        "matched_album_name": best["album_name"],
        "matched_release_year": best["release_year"],
        "fuzzy_score": best_score,
    }


unmatched = exact_matches[exact_matches["track_id"].isna()].copy()

fuzzy_rows = []
for _, row in unmatched.iterrows():
    result = best_fuzzy_match(row["normalized_history_title"], atlas_titles)
    fuzzy_rows.append(result)

fuzzy_df = pd.DataFrame(fuzzy_rows, index=unmatched.index)

for col in fuzzy_df.columns:
    unmatched[col] = fuzzy_df[col]

unmatched["match_type"] = unmatched["fuzzy_score"].apply(
    lambda x: "fuzzy" if x >= 0.86 else "unmatched"
)

unmatched[
    [
        "master_metadata_album_artist_name",
        "master_metadata_track_name",
        "hours_played",
        "matched_track_name",
        "fuzzy_score",
        "match_type",
    ]
].sort_values("hours_played", ascending=False).head(50)

,master_metadata_album_artist_name,master_metadata_track_name,hours_played,matched_track_name,fuzzy_score,match_type
16,Jung Kook,Stay Alive (Prod. SUGA of BTS),1.671669,Stay - Live,0.947368,fuzzy
24,Jung Kook,Still With You,1.455822,SWIM with SUGA (Melodic Techno Remix),0.642857,unmatched
33,j-hope,Killin' It Girl (Solo Version),1.194645,SWIM with RM (Chill Hip Hop Remix),0.461538,unmatched
37,BTS,Come Over,1.071254,Come Back Home,0.608696,unmatched
48,Jung Kook,Standing Next to You,0.948286,Intro: What Am I to You,0.523810,unmatched
59,j-hope,Killin' It Girl (feat. GloRilla),0.673744,SWIM with RM (Chill Hip Hop Remix),0.461538,unmatched
74,j-hope,MONA LISA,0.575173,ON - Live,0.625000,unmatched
80,Agust D,Daechwita,0.540359,Ma City,0.500000,unmatched
85,Jung Kook,Seven (feat. Latto) (Explicit Ver.),0.486995,Serendipity (Full Length Edition),0.500000,unmatched
94,Jung Kook,Closer to You (feat. Major Lazer),0.391847,For You,0.600000,unmatched


In [17]:
exact_ready = exact_matches[exact_matches["track_id"].notna()].copy()

exact_ready["matched_track_id"] = exact_ready["track_id"]
exact_ready["matched_track_name"] = exact_ready["track_name"]
exact_ready["matched_album_name"] = exact_ready["album_name"]
exact_ready["matched_release_year"] = exact_ready["release_year"]
exact_ready["fuzzy_score"] = 1.0

accepted_fuzzy = unmatched[unmatched["match_type"] == "fuzzy"].copy()

matched_history = pd.concat(
    [
        exact_ready[
            [
                "master_metadata_album_artist_name",
                "master_metadata_track_name",
                "master_metadata_album_album_name",
                "plays",
                "hours_played",
                "total_ms",
                "first_play",
                "last_play",
                "matched_track_id",
                "matched_track_name",
                "matched_album_name",
                "matched_release_year",
                "match_type",
                "fuzzy_score",
            ]
        ],
        accepted_fuzzy[
            [
                "master_metadata_album_artist_name",
                "master_metadata_track_name",
                "master_metadata_album_album_name",
                "plays",
                "hours_played",
                "total_ms",
                "first_play",
                "last_play",
                "matched_track_id",
                "matched_track_name",
                "matched_album_name",
                "matched_release_year",
                "match_type",
                "fuzzy_score",
            ]
        ],
    ],
    ignore_index=True,
)

matched_history.shape

(501, 14)

In [18]:
personal_track_stats = (
    matched_history
    .groupby("matched_track_id", dropna=False)
    .agg(
        personal_plays=("plays", "sum"),
        personal_hours=("hours_played", "sum"),
        personal_total_ms=("total_ms", "sum"),
        personal_first_play=("first_play", "min"),
        personal_last_play=("last_play", "max"),
        source_history_titles=("master_metadata_track_name", lambda x: " | ".join(sorted(set(x)))),
        match_types=("match_type", lambda x: " | ".join(sorted(set(x)))),
        min_fuzzy_score=("fuzzy_score", "min"),
    )
    .reset_index()
    .rename(columns={"matched_track_id": "track_id"})
)

In [19]:
def mastery_level(hours: float) -> int:
    if hours <= 0:
        return 0
    if hours < 0.5:
        return 1
    if hours < 1:
        return 2
    if hours < 2:
        return 3
    if hours < 5:
        return 4
    if hours < 10:
        return 5
    if hours < 20:
        return 6
    if hours < 40:
        return 7
    if hours < 75:
        return 8
    if hours < 150:
        return 9
    return 10


personal_track_stats["personal_rank"] = (
    personal_track_stats["personal_hours"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

personal_track_stats["mastery_level"] = personal_track_stats["personal_hours"].apply(mastery_level)

In [20]:
personal_overlay = atlas_df.merge(
    personal_track_stats,
    on="track_id",
    how="left",
)

personal_overlay["personal_plays"] = personal_overlay["personal_plays"].fillna(0).astype(int)
personal_overlay["personal_hours"] = personal_overlay["personal_hours"].fillna(0)
personal_overlay["personal_total_ms"] = personal_overlay["personal_total_ms"].fillna(0).astype(int)
personal_overlay["mastery_level"] = personal_overlay["mastery_level"].fillna(0).astype(int)
personal_overlay["has_listening_history"] = personal_overlay["personal_plays"] > 0

personal_overlay[
    ["track_name", "album_name", "personal_hours", "personal_plays", "personal_rank", "mastery_level"]
].sort_values("personal_hours", ascending=False).head(30)

,track_name,album_name,personal_hours,personal_plays,personal_rank,mastery_level
300,Tomorrow,Skool Luv Affair,3.837981,102,1.0,4
291,Tomorrow,Skool Luv Affair (Special Addition),3.837981,102,1.0,4
363,MIC Drop (feat. Desiigner) [Steve Aoki Remix],MIC Drop (feat. Desiigner) [Steve Aoki Remix],2.375706,63,2.0,4
144,MIC Drop,Love Yourself 結 'Answer',2.375706,63,2.0,4
147,MIC Drop (Steve Aoki Remix) (Full Length Edition),Love Yourself 結 'Answer',2.375706,63,2.0,4
176,MIC Drop,Love Yourself 承 'Her',2.375706,63,2.0,4
276,Let Me Know,Dark & Wild,2.312484,50,3.0,4
175,Pied Piper,Love Yourself 承 'Her',1.935010,55,4.0,3
239,House of Cards (Full Length Edition),The Most Beautiful Moment in Life: Young Forever,1.917402,39,5.0,3
359,FAKE LOVE (Rocking Vibe Mix),FAKE LOVE (Rocking Vibe Mix),1.790389,108,6.0,3


In [23]:
personal_overlay_path = PERSONAL_DIR / "personal_atlas_overlay.parquet"
matched_history_path = PERSONAL_DIR / "matched_listening_history.parquet"

personal_overlay.to_parquet(personal_overlay_path, index=False)
matched_history.to_parquet(matched_history_path, index=False)

personal_overlay_path, matched_history_path

(PosixPath('/home/seans/becode/projects/BTS-Song-Atlas-/data/processed/personal_atlas/personal_atlas_overlay.parquet'),
 PosixPath('/home/seans/becode/projects/BTS-Song-Atlas-/data/processed/personal_atlas/matched_listening_history.parquet'))

In [24]:
summary = {
    "history_songs": len(bts_song_league),
    "matched_history_rows": len(matched_history),
    "atlas_tracks": len(atlas_df),
    "atlas_tracks_with_history": personal_overlay["has_listening_history"].sum(),
    "total_personal_hours_matched": personal_overlay["personal_hours"].sum(),
}

summary

{'history_songs': 344,
 'matched_history_rows': 501,
 'atlas_tracks': 381,
 'atlas_tracks_with_history': np.int64(313),
 'total_personal_hours_matched': np.float64(120.20760000000001)}

In [25]:
personal_overlay[
    ["track_name", "album_name", "personal_hours", "personal_plays", "mastery_level"]
].sort_values("personal_hours", ascending=False).head(30)

,track_name,album_name,personal_hours,personal_plays,mastery_level
300,Tomorrow,Skool Luv Affair,3.837981,102,4
291,Tomorrow,Skool Luv Affair (Special Addition),3.837981,102,4
363,MIC Drop (feat. Desiigner) [Steve Aoki Remix],MIC Drop (feat. Desiigner) [Steve Aoki Remix],2.375706,63,4
144,MIC Drop,Love Yourself 結 'Answer',2.375706,63,4
147,MIC Drop (Steve Aoki Remix) (Full Length Edition),Love Yourself 結 'Answer',2.375706,63,4
176,MIC Drop,Love Yourself 承 'Her',2.375706,63,4
276,Let Me Know,Dark & Wild,2.312484,50,4
175,Pied Piper,Love Yourself 承 'Her',1.935010,55,3
239,House of Cards (Full Length Edition),The Most Beautiful Moment in Life: Young Forever,1.917402,39,3
359,FAKE LOVE (Rocking Vibe Mix),FAKE LOVE (Rocking Vibe Mix),1.790389,108,3


In [26]:
matched_history.query("match_type == 'fuzzy'")[
    ["master_metadata_track_name", "matched_track_name", "fuzzy_score", "hours_played"]
].sort_values("fuzzy_score").head(30)

,master_metadata_track_name,matched_track_name,fuzzy_score,hours_played
500,Come back to me,Come Back Home,0.896552,0.000597
499,Stay Alive (Prod. SUGA of BTS),Stay - Live,0.947368,1.671669


## Canonical Matching

In [28]:
DATA_PROCESSED = Path("../../data/processed")
PERSONAL_DIR = DATA_PROCESSED / "personal_atlas"

atlas_df = pd.read_parquet(DATA_PROCESSED / "song_embeddings.parquet")
league = pd.read_parquet(PERSONAL_DIR / "bts_song_league.parquet")

print(atlas_df.columns.tolist()[:30])
print(atlas_df.shape)
print(league.shape)

['track_id', 'track_name', 'album_name', 'release_year', 'embedding_0', 'embedding_1', 'embedding_2', 'embedding_3', 'embedding_4', 'embedding_5', 'embedding_6', 'embedding_7', 'embedding_8', 'embedding_9', 'embedding_10', 'embedding_11', 'embedding_12', 'embedding_13', 'embedding_14', 'embedding_15', 'embedding_16', 'embedding_17', 'embedding_18', 'embedding_19', 'embedding_20', 'embedding_21', 'embedding_22', 'embedding_23', 'embedding_24', 'embedding_25']
(381, 388)
(344, 8)


In [29]:
[col for col in atlas_df.columns if "canon" in col.lower() or "title" in col.lower() or "name" in col.lower()]

['track_name', 'album_name']

In [30]:
atlas_df[["track_name", "album_name"]].head(20)

,track_name,album_name
0,SWIM,KEEP SWIMMING
1,SWIM with RM (Chill Hip Hop Remix),KEEP SWIMMING
2,SWIM with Jin (Alternative Rock Remix),KEEP SWIMMING
3,SWIM with SUGA (Melodic Techno Remix),KEEP SWIMMING
4,SWIM with j-hope (Afrobeat Remix),KEEP SWIMMING
5,SWIM with Jimin (Slow Jam R&B Remix),KEEP SWIMMING
6,SWIM with V (Electronic Remix),KEEP SWIMMING
7,SWIM with Jung Kook (Acoustic Lofi Remix),KEEP SWIMMING
8,Body to Body,ARIRANG
9,Hooligan,ARIRANG


In [31]:
def canonicalize_title(title):
    if pd.isna(title):
        return ""

    title = str(title).lower()

    title = re.sub(r"\(.*?\)", "", title)
    title = re.sub(r"\[.*?\]", "", title)

    remove_phrases = [
        "full length edition",
        "steve aoki remix",
        "rocking vibe mix",
        "urban mix",
        "remix",
        "remastered",
        "remaster",
        "live",
        "feat desiigner",
        "feat.",
        "feat",
    ]

    for phrase in remove_phrases:
        title = title.replace(phrase, "")

    title = title.replace("&", "and")
    title = re.sub(r"[^a-z0-9가-힣ぁ-んァ-ン一-龯\s]", " ", title)
    title = re.sub(r"\s+", " ", title).strip()

    return title

atlas_df["canonical_key"] = atlas_df["track_name"].apply(canonicalize_title)
league["history_key"] = league["master_metadata_track_name"].apply(canonicalize_title)

In [32]:
canonical_atlas = (
    atlas_df
    .groupby("canonical_key", as_index=False)
    .agg(
        canonical_title=("track_name", "first"),
        representative_album=("album_name", "first"),
        version_count=("track_name", "count"),
        atlas_versions=("track_name", lambda x: sorted(set(x))),
        atlas_albums=("album_name", lambda x: sorted(set(x))),
    )
)

canonical_atlas.sort_values("version_count", ascending=False).head(30)

,canonical_key,canonical_title,representative_album,version_count,atlas_versions,atlas_albums
56,dynamite,Dynamite - Live,PERMISSION TO DANCE ON STAGE - LIVE,14,"[Dynamite, Dynamite (Acoustic Remix), Dynamite...","[BE, Dynamite (DayTime Version), Dynamite (Hol..."
37,butter,Butter - Live,PERMISSION TO DANCE ON STAGE - LIVE,9,"[Butter, Butter (Cooler Remix), Butter (Holida...","[Butter (Holiday Remix), Butter (Hotter, Sweet..."
61,fake love,FAKE LOVE - Live,PERMISSION TO DANCE ON STAGE - LIVE,6,"[FAKE LOVE, FAKE LOVE (Rocking Vibe Mix), FAKE...","[FAKE LOVE (Rocking Vibe Mix), Love Yourself 結..."
51,dna,DNA - Live,PERMISSION TO DANCE ON STAGE - LIVE,5,"[DNA, DNA (Pedal 2 LA Mix), DNA - Live]","[Love Yourself 承 'Her', Love Yourself 結 'Answe..."
83,i need u,I NEED U,Proof,5,"[I NEED U, I Need U, I Need U (Remix), I Need ...","[Proof, The Most Beautiful Moment in Life Pt.1..."
169,run,RUN,Proof,5,"[RUN, Run, Run (Alternative Mix), Run (Ballad ...","[Proof, The Most Beautiful Moment in Life Pt.2..."
85,idol,IDOL - Live,PERMISSION TO DANCE ON STAGE - LIVE,4,"[IDOL, IDOL - Live]","[Love Yourself 結 'Answer', PERMISSION TO DANCE..."
23,blood sweat and tears,Blood Sweat & Tears - Live,PERMISSION TO DANCE ON STAGE - LIVE,4,"[Blood Sweat & Tears, Blood Sweat & Tears - Live]","[PERMISSION TO DANCE ON STAGE - LIVE, Proof, W..."
29,boy with luv,Boy With Luv (Feat. Halsey) - Live,PERMISSION TO DANCE ON STAGE - LIVE,4,"[Boy With Luv (Feat. Halsey), Boy With Luv (Fe...","[MAP OF THE SOUL : 7, MAP OF THE SOUL : PERSON..."
163,permission to dance,Permission to Dance - Live,PERMISSION TO DANCE ON STAGE - LIVE,4,"[Permission to Dance, Permission to Dance (R&B...","[Butter / Permission to Dance, PERMISSION TO D..."


In [33]:
matched = league.merge(
    canonical_atlas,
    left_on="history_key",
    right_on="canonical_key",
    how="left",
)

matched["match_type"] = matched["canonical_title"].notna().map({
    True: "exact",
    False: "unmatched"
})

matched["match_type"].value_counts()

match_type
exact        247
unmatched     97
Name: count, dtype: int64

In [34]:
matched[matched["match_type"] == "unmatched"][
    [
        "master_metadata_album_artist_name",
        "master_metadata_track_name",
        "master_metadata_album_album_name",
        "hours_played",
        "plays",
        "history_key",
    ]
].sort_values("hours_played", ascending=False).head(50)


,master_metadata_album_artist_name,master_metadata_track_name,master_metadata_album_album_name,hours_played,plays,history_key
8,Jung Kook,Stay Alive (Prod. SUGA of BTS),Stay Alive (Prod. SUGA of BTS),1.671669,81,stay a
12,Jung Kook,Still With You,Still With You,1.455822,48,still with you
15,j-hope,Killin' It Girl (Solo Version),Killin' It Girl (feat. GloRilla),1.194645,54,killin it girl
18,BTS,Come Over,Come Over,1.071254,36,come over
25,Jung Kook,Standing Next to You,GOLDEN,0.948286,27,standing next to you
33,j-hope,Killin' It Girl (feat. GloRilla),Killin' It Girl (feat. GloRilla),0.673744,21,killin it girl
39,j-hope,MONA LISA,MONA LISA,0.575173,23,mona lisa
44,Agust D,Daechwita,D-2,0.540359,18,daechwita
47,Jung Kook,Seven (feat. Latto) (Explicit Ver.),GOLDEN,0.486995,13,seven
53,Jung Kook,Closer to You (feat. Major Lazer),GOLDEN,0.391847,21,closer to you


In [35]:
print(f"History songs: {len(league)}")
print(f"Matched: {matched['canonical_title'].notna().sum()}")
print(f"Coverage: {matched['canonical_title'].notna().mean():.1%}")

print("\nTop unmatched artists:")
print(
    matched.loc[matched["canonical_title"].isna(), "master_metadata_album_artist_name"]
    .value_counts()
)

History songs: 344
Matched: 247
Coverage: 71.8%

Top unmatched artists:
master_metadata_album_artist_name
BTS          21
j-hope       18
Agust D      17
Jung Kook    16
Jimin        10
V             9
RM            5
Jin           1
Name: count, dtype: int64


In [36]:
matched[
    (matched["canonical_title"].isna()) &
    (matched["master_metadata_album_artist_name"] == "BTS")
][
    [
        "master_metadata_track_name",
        "master_metadata_album_album_name",
        "hours_played"
    ]
].sort_values("hours_played", ascending=False)

,master_metadata_track_name,master_metadata_album_album_name,hours_played
18,Come Over,Come Over,1.071254
92,No. 29,ARIRANG,0.141079
93,Embarrassed,Dark & Wild,0.138690
103,Paradise,Love Yourself 轉 'Tear',0.119411
214,Dream Glow (BTS World Original Soundtrack) (Pt...,Dream Glow (BTS World Original Soundtrack) (Pt...,0.031360
228,Interlude,2 Cool 4 Skool,0.014567
233,2.0 (Live at Pier 17) | Presented by Spotify,2.0 (Live at Pier 17) | Presented by Spotify,0.010632
241,Skit: One Night in a Strange City,The Most Beautiful Moment in Life Pt.2,0.005556
246,Skit: Billboard Music Awards Speech,Love Yourself 承 'Her',0.004096
257,Heartbeat (BTS World Original Soundtrack),BTS WORLD (Original Soundtrack),0.002787


In [37]:
EXCLUDED_TRACK_TYPES = {
    "skit",
    "interlude",
}

EXCLUDED_TRACKS = {
    "2.0 (Live at Pier 17) | Presented by Spotify",
}

In [38]:
def should_ignore(track_name):
    lower = track_name.lower()

    if "skit" in lower:
        return True

    if "interlude" in lower:
        return True

    if track_name in EXCLUDED_TRACKS:
        return True

    return False

# Conclusion

## Summary

This notebook successfully links Spotify Extended Streaming History with the current BTS Song Atlas dataset.

### Results

- History songs: 344
- Matched songs: 247
- Coverage: 71.8%

### Findings

Most unmatched songs are **not matching errors**.

They primarily belong to:

- Solo discographies (planned for v1.1)
- BTS songs not yet included in the atlas
- Intentional exclusions such as skits and special live releases

The matching pipeline is therefore considered production-ready for the current scope.

Future atlas expansions will automatically improve coverage without requiring changes to the matching logic.

## Outputs

Generated:

- spotify_history_clean.parquet
- bts_listening_history.parquet
- bts_song_league.parquet
- bts_album_league.parquet
- matched_listening_history.parquet
- personal_atlas_overlay.parquet

In [41]:
from pathlib import Path

OUT_DIR = Path("../../data/processed/personal_atlas")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Final matched listening history
matched.to_parquet(
    OUT_DIR / "matched_listening_history.parquet",
    index=False,
)

# One row per atlas song with personal stats
personal_overlay.to_parquet(
    OUT_DIR / "personal_atlas_overlay.parquet",
    index=False,
)

# Canonical atlas lookup (useful for future notebooks)
canonical_atlas.to_parquet(
    OUT_DIR / "canonical_artist_lookup.parquet",
    index=False,
)

print("✅ Saved:")
print(f"  • {OUT_DIR / 'matched_listening_history.parquet'}")
print(f"  • {OUT_DIR / 'personal_atlas_overlay.parquet'}")
print(f"  • {OUT_DIR / 'canonical_song_lookup.parquet'}")

✅ Saved:
  • ../../data/processed/personal_atlas/matched_listening_history.parquet
  • ../../data/processed/personal_atlas/personal_atlas_overlay.parquet
  • ../../data/processed/personal_atlas/canonical_song_lookup.parquet
